In [2]:
# Cell 1: Install & import libs (FIXED)
import sys
import subprocess
import os

def install_package(package):
    """Safely install a package using pip"""
    try:
        # Try using pip directly
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install {package}: {e}")
        return False

def check_package(package_name, import_name=None):
    """Check if a package is already installed"""
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
        print(f"✅ {package_name} is already installed")
        return True
    except ImportError:
        print(f"⚠️  {package_name} not found, installing...")
        return False

print("🔧 Checking and installing required packages...")

# Check and install packages
packages = [
    ("pandas", "pandas"),
    ("openai", "openai"),
    ("scikit-learn", "sklearn")
]

for package, import_name in packages:
    if not check_package(package, import_name):
        install_package(package)

print("\n📦 Importing libraries...")
try:
    import pandas as pd
    import openai
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Try restarting the kernel and running this cell again")

print("\n🎯 Ready to proceed!")

🔧 Checking and installing required packages...
✅ pandas is already installed
✅ openai is already installed
✅ scikit-learn is already installed

📦 Importing libraries...
✅ All imports successful!

🎯 Ready to proceed!


In [3]:
# System diagnostic and package version checker
import sys
import platform
import os

print("🔍 SYSTEM DIAGNOSTICS")
print("=" * 40)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Architecture: {platform.architecture()}")
print(f"Python executable: {sys.executable}")
print(f"Current working directory: {os.getcwd()}")

print("\n📦 PACKAGE VERSIONS:")
packages_to_check = {
    'pandas': 'pd',
    'openai': 'openai', 
    'sklearn': 'sklearn',
    'numpy': 'np'
}

for package, alias in packages_to_check.items():
    try:
        if package == 'pandas':
            import pandas as pd
            print(f"✅ pandas: {pd.__version__}")
        elif package == 'openai':
            import openai
            print(f"✅ openai: {openai.__version__}")
        elif package == 'sklearn':
            import sklearn
            print(f"✅ scikit-learn: {sklearn.__version__}")
        elif package == 'numpy':
            import numpy as np
            print(f"✅ numpy: {np.__version__}")
    except ImportError:
        print(f"❌ {package}: Not installed")
    except Exception as e:
        print(f"⚠️  {package}: Error - {e}")

print("\n🎯 All systems ready for multi-label classification!")

# List of labels
labels = ["Medical", "Mental Health", "Abuse", "Aggression", "Sexual", "Discrimination", "Pregnancy", "Not Applicable"]

# Updated definitions (unchanged for now)
definitions = {
    "Medical": "The post should be assigned this label if it talks about taking medical pills, going through procedures or treatments related to pregnancy loss (like abortion or miscarriage), or getting medical care after sexual violence. It also includes mentions of bleeding, blood, or anything graphic like gore related to pregnancy loss or sexual violence.",
    
    "Mental Health": "The post should be assigned this label if it talks about emotional pain like sadness, crying, anxiety, panic, or stress.",
    
    "Abuse": "The post should be assigned this label if someone is being controlled, threatened, or harmed emotionally or physically by another person. This includes situations where a person is forced to do something, made to feel unsafe, or has their freedom or privacy violated. It includes coercion, manipulation, intimidation, stalking, voyeurism, rape, hate speech, and all forms of sexual violence.",
    
    "Aggression": "The post should be assigned this label if it describes someone being physically attacked, threatened, or facing violent behavior during or after a traumatic event. It includes things like hitting, pushing, chasing, restraining, or threats of harm.",
    
    "Sexual": "The post should be assigned this label if it includes sexual content such as sex acts, sexual behavior, pornography, kinks, or sexual preferences, whether harmful or not. This also includes kinks that do not involve sex acts, like praise kink or size kink — they still count as sexual content.",
    
    "Discrimination": "The post should be assigned this label when someone is judged, mistreated, or shamed because of their gender, body, identity, or background.",
    
    "Pregnancy": "The post should be assigned this label if it talks about pregnancy loss, like miscarriage, abortion, or stillbirth, or complications related to pregnancy. It should not be used for general pregnancy or childbirth unless it is clearly connected to a loss or medical issue.",
    
    "Not Applicable": "Used when the post doesn't match any of the above trigger categories or topics."
}


🔍 SYSTEM DIAGNOSTICS
Python version: 3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]
Platform: Linux-6.8.0-1035-aws-x86_64-with-glibc2.35
Architecture: ('64bit', 'ELF')
Python executable: /bin/python3
Current working directory: /home/ubuntu

📦 PACKAGE VERSIONS:
✅ pandas: 2.3.1
✅ openai: 1.99.1
✅ scikit-learn: 1.7.1
✅ numpy: 2.2.6

🎯 All systems ready for multi-label classification!


In [3]:
# Load Dataset
df = pd.read_csv("Combined_Dataset_Annotations - Combined_Dataset.csv")
df.head()

,id,soure,subreddit,title,body,created_utc,url,Tags
0,1ljxynj,abortion,abortion,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...",2025-06-25 5:59:34,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Sexual, Pregnancy"
1,1ljxtt8,NaN,abortion,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,2025-06-25 5:51:08,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy, Mental Health"
2,1ljwhkb,abortion,abortion,Help needed/ live in Texas where abortion in b...,Anyone know of a legit site to support women i...,2025-06-25 4:31:43,https://www.reddit.com/r/abortion/comments/1lj...,"Discrimination, Pregnancy"
3,1ljvy6u,NaN,abortion,medical abortion at 6 weeks,I’ll be doing my procedure on Friday and I got...,2025-06-25 4:02:28,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy"
4,1ljv5k4,abortion,abortion,Idk what to feel about my decision after doing...,I just had medical abortion yesterday. I was a...,2025-06-25 3:19:54,https://www.reddit.com/r/abortion/comments/1lj...,"Pregnancy, Mental Health"


In [4]:
# 🛠️ Step 1: Clean 'Tags' column if needed
df["Tags"] = df["Tags"].apply(
    lambda x: [tag.strip() for tag in x.split(",")] if isinstance(x, str) else x
)

# 🧷 Step 2: Create binary columns per label
all_labels = [
    "Medical",
    "Mental Health",
    "Abuse",
    "Aggression",
    "Sexual",
    "Discrimination",
    "Pregnancy",
    "Not Applicable"
]

for label in all_labels:
    df[label] = df["Tags"].apply(lambda tags: int(label in tags if isinstance(tags, list) else []))


In [5]:
print(df.columns.tolist())


['id', 'soure', 'subreddit', 'title', 'body', 'created_utc', 'url', 'Tags', 'Medical', 'Mental Health', 'Abuse', 'Aggression', 'Sexual', 'Discrimination', 'Pregnancy', 'Not Applicable']


In [6]:
# Cell 6 : Prompt 2

def make_per_label_prompt(title, body, label_name, definition):
    """
    Create a chain-of-thought prompt for evaluating a single label.

    Arguments:
    - title: Reddit post title
    - body: Reddit post description
    - label_name: The label we are testing (e.g., "Medical", "Abuse")
    - definition: The definition text for this label

    Returns:
    - Formatted prompt string for the LLM
    """

    prompt = (
    "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
    "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
    "You should analyze each post and evaluate whether readers could be triggered by it. "
    "We have multiple categories of trigger warnings that you can provide, and in this case, "
    "You will evaluate **only one trigger label at a time** using a step by step approach.\n\n"
    f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
    f"Definition of '{label_name}':\n"
    f"{definition.strip()}\n\n"
    "Step-by-step instructions:\n"
    "1. Carefully read the post (Title and Description).\n"
    "2. Think step by step about whether the post *clearly matches* the above definition.\n"
    "3. Be cautious — if the post is vague or not directly related to the label, answer NO.\n"
    "4. If there is strong evidence that matches the definition, answer YES.\n"
    "5. Do not assume or guess based on weak signals.\n\n"
    f"Title: {title.strip()}\n\n"
    f"Description: {body.strip()}\n\n"
    "Answer with only YES or NO.\n"
    f"Does this post contain the '{label_name}' trigger?\n"
    "Answer:"
)

    return prompt


In [6]:
import json
with open("similar_posts_k10.json", "r") as f:
    similar_post_map = json.load(f)

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
import json
import pandas as pd

# ---- Globals / setup (same assumptions as your code) ----
global collected_examples_per_label
collected_examples_per_label = {}  # Reset at each call

# Assumes `labels` (list of label names) and `definitions` (dict[label] -> text) are already defined
df = pd.read_csv("annotations_dataset_with_index.csv")
embeddings = torch.load("post_embeddings.pt")
model = SentenceTransformer("all-mpnet-base-v2")
with open("similar_posts_k10.json", "r") as f:
    similar_post_map = json.load(f)


def make_multishot_prompt_cosine(k, title, body, current_index=None, include_defs=True, debug=False):
    """
    Builds one prompt per label using the EXACT template the user provided.
    Also computes top-k similar positive examples per label to populate
    `collected_examples_per_label` for external inspection, but DOES NOT
    insert examples into the prompt (to keep it exact).
    Returns: dict[label] -> prompt_str
    """
    global collected_examples_per_label
    collected_examples_per_label = {}  # reset on each call

    input_text = (title or "") + " " + (body or "")
    input_embedding = model.encode([input_text])[0]

    prompts_by_label = {}

    for label_name in labels:
        # ---- Gather positive examples for this label (for external use only) ----
        if label_name not in df.columns:
            # Keep structure predictable even if column missing
            collected_examples_per_label[label_name] = []
            if debug:
                print(f"⚠️ Skipping label '{label_name}' — column not found in dataframe.")
        else:
            trigger_examples = df[df[label_name].fillna(0) == 1]
            if current_index is not None:
                trigger_examples = trigger_examples[trigger_examples.index != current_index]

            selected_indices, selected_scores = [], []

            # Case 1: Use precomputed similar indices from JSON (if available for this post)
            used_json = False
            if current_index is not None and str(current_index) in similar_post_map:
                used_json = True
                similar_ids_with_scores = [(int(idx), float(score)) for idx, score in similar_post_map[str(current_index)]]
                for idx, score in similar_ids_with_scores:
                    if idx in trigger_examples.index:
                        selected_indices.append(idx)
                        selected_scores.append(score)
                    if len(selected_indices) == k:
                        break

            # Case 2: Compute similarity if needed
            if not selected_indices and not trigger_examples.empty:
                example_indices = trigger_examples.index.tolist()
                example_embeddings = embeddings[example_indices]
                sims = cosine_similarity([input_embedding], example_embeddings)[0]
                top_k_idx = np.argsort(sims)[-k:][::-1]
                selected_indices = [example_indices[i] for i in top_k_idx]
                selected_scores = [float(sims[i]) for i in top_k_idx]

            # Store for external consumption
            collected_examples_per_label[label_name] = []
            for idx, score in zip(selected_indices, selected_scores):
                row = df.loc[idx]
                collected_examples_per_label[label_name].append({
                    "index": int(idx),
                    "score": round(score, 4),
                    "title": row.get("title", "[No Title]"),
                    "body": row.get("body", "[No Body]")
                })
                if debug:
                    print(f"\n🔎 Example for '{label_name}' "
                          f"(source={'json' if used_json else 'cosine'}): idx={idx} score={score:.4f}")
                    print(f"• Title: {row.get('title', '[No Title]')}")
                    print(f"• Body: {row.get('body', '[No Body]')[:300]}...")
                    print("-" * 80)

        # ---- Build the EXACT prompt string for this label ----
        definition = definitions.get(label_name, "").strip()
        # Note: do not add or alter wording; keep exactly as provided
        prompt_str = (
            "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
            "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
            "Your task is to analyze the post and decide whether readers could be triggered by it. "
            "We have multiple categories of trigger warnings, but in this case, "
            "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
            f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
            f"Definition of '{label_name}':\n"
            f"{definition}\n\n"
            "Step-by-step instructions:\n"
            "1. Carefully read the post (Title and Description).\n"
            "2. Decide whether the post *explicitly and clearly matches* the above definition.\n"
            "3. If the post is vague, uncertain, or not directly related to the label, answer NO.\n"
            "4. If there is strong, explicit evidence that matches the definition, answer YES.\n"
            "5. Do not infer, assume, or guess based on weak signals.\n\n"
            f"Title: {str(title).strip()}\n\n"
            f"Description: {str(body).strip()}\n\n"
            "Answer with only YES or NO (uppercase, no punctuation, no explanation).\n"
            f"Does this post contain the '{label_name}' trigger?\n"
            "Answer:"
        )

        prompts_by_label[label_name] = prompt_str

    return prompts_by_label


# (Optional) Convenience helper if you want a single label’s prompt directly
def make_prompt_for_label(label_name, k, title, body, current_index=None, debug=False):
    pm = make_multishot_prompt_cosine(k, title, body, current_index=current_index, debug=debug)
    return pm[label_name]


In [9]:
def parse_trigger_labels(llm_response):
    """
    Convert LLM YES/NO responses to 0/1 values for each trigger label.
    Assumes answers are given in the format:
    Is it a possible trigger for [Label]? YES/NO
    """
    result = {}
    for label in labels:
        # Create a matching string like "trigger for Abuse"
        keyword = f"trigger for {label.lower()}"
        # Search line-by-line
        for line in llm_response.splitlines():
            if keyword in line.lower():
                if "yes" in line.lower():
                    result[label] = 1
                else:
                    result[label] = 0
                break
        else:
            # If label line isn't found at all, mark it as 0
            result[label] = 0
    return result

In [10]:
# OpenAI Version Check and Compatibility Fix
import openai
import os
from dotenv import load_dotenv  # ✅ Correct syntax
from packaging import version

# Load environment variables from .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("🔍 Checking OpenAI configuration...")
print(f"OpenAI version: {openai.__version__}")

# Determine API format using version
if version.parse(openai.__version__) >= version.parse("1.0.0"):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    api_format = "modern"
    print("✅ Using modern OpenAI API format (v1.x)")
else:
    openai.api_key = OPENAI_API_KEY
    api_format = "legacy"
    print("✅ Using legacy OpenAI API format (v0.x)")

print(f"🔧 API Format: {api_format}")
print("🎯 OpenAI setup complete!")


🔍 Checking OpenAI configuration...
OpenAI version: 1.99.1
✅ Using modern OpenAI API format (v1.x)
🔧 API Format: modern
🎯 OpenAI setup complete!


In [11]:
from openai import OpenAI
from dotenv import load_dotenv

# Load API key from .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)
print([m.id for m in client.models.list().data])


['gpt-4-0613', 'gpt-4', 'gpt-3.5-turbo', 'gpt-5-nano', 'gpt-5', 'gpt-5-mini-2025-08-07', 'gpt-5-mini', 'gpt-5-nano-2025-08-07', 'davinci-002', 'babbage-002', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo-instruct-0914', 'dall-e-3', 'dall-e-2', 'gpt-4-1106-preview', 'gpt-3.5-turbo-1106', 'tts-1-hd', 'tts-1-1106', 'tts-1-hd-1106', 'text-embedding-3-small', 'text-embedding-3-large', 'gpt-4-0125-preview', 'gpt-4-turbo-preview', 'gpt-3.5-turbo-0125', 'gpt-4-turbo', 'gpt-4-turbo-2024-04-09', 'gpt-4o', 'gpt-4o-2024-05-13', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini', 'gpt-4o-2024-08-06', 'chatgpt-4o-latest', 'o1-mini-2024-09-12', 'o1-mini', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-audio-preview', 'gpt-4o-realtime-preview', 'omni-moderation-latest', 'omni-moderation-2024-09-26', 'gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4o-mini-audio-preview-2024-12-17', 'o1-2024-12-17', 'o1', 'gpt-4o-

In [12]:
# --- Helper that supports binary vs structured modes ---
import time, random, os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

DEFAULT_MODEL = "gpt-4.1-mini"  # or "gpt-4o-mini"

def getTriggerWarningsLLMResponse(
    post_text: str,
    model: str = DEFAULT_MODEL,
    mode: str = "structured",     # <-- "structured" for few-shot; "binary" only for single-label YES/NO
    temperature: float = 0.0,
    max_tokens: int = 512,        # <-- need room for per-label answers
    verbose: bool = True,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 20.0,
) -> str:
    if verbose:
        print("🧠 Classifying post for trigger warnings...")
        print("=" * 60)

    # Choose system instruction by mode
    if mode == "binary":
        system_msg = "Answer ONLY with YES or NO. No punctuation, no explanation."
    else:
        system_msg = (
            "Follow the user’s instructions exactly. For EACH label block you see, write a line "
            "in the form `Answer: YES` or `Answer: NO`. At the very end, include a final line in the form "
            "`Labels: [<comma-separated list of applicable labels or Not Applicable>]`. Do not omit any labels."
        )

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": post_text},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        except Exception as e:
            # Respect Retry-After if present
            retry_after = None
            try:
                retry_after = float(getattr(getattr(e, "response", None), "headers", {}).get("retry-after", 0))
            except Exception:
                pass

            msg = str(e)
            if verbose:
                print(f"⚠️ Error: {msg}")

            low = msg.lower()
            if "rate limit" in low or "429" in low:
                wait = retry_after if retry_after and retry_after > 0 else min(base_delay * (2 ** attempt), max_delay)
                wait *= (0.8 + 0.4 * random.random())
                if verbose:
                    print(f"⏳ Rate limit; retrying in {wait:.2f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            if verbose:
                print("❌ Non-retryable error; returning empty string.")
            return ""

    if verbose:
        print("❌ Gave up after retries; returning empty string.")
    return ""


In [13]:
# Cell 3: Chain-of-thought prompting for each label
import time
import pandas as pd

# Sample post for evaluation
title = "Relationship fight and sudden bleeding"
sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night."
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)
    response = getTriggerWarningsLLMResponse(single_prompt)
    
    answer = response.strip().upper() if response else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []

for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")



🧠 Evaluating Label: Medical


NameError: name 'make_per_label_prompt' is not defined

In [31]:
# Cell: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Assumes these already exist in your environment:
# - labels: List[str] (includes "Not Applicable" or not — both fine)
# - definitions: Dict[str, str]
# - make_per_label_prompt(title, body, label, definition)
# - getTriggerWarningsLLMResponse(prompt)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_prompt_1.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return final labels + per-label decisions."""
    per_label = {}
    for label in labels:
        definition = definitions[label]
        prompt = make_per_label_prompt(title, body, label, definition)
        resp = getTriggerWarningsLLMResponse(prompt)
        per_label[label] = normalize_yes_no(resp)
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Build final labels: if none are YES (excluding NA), mark Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
df = df[['title', 'body']].fillna('').astype(str)

subset = df.head(75).copy()

predicted_lists = []
per_label_json  = []

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))
    print(f"{i}. Labels: {final_labels}")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nSaved predictions to {OUTPUT_CSV}")


🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...
1. Labels: ['Medical', 'Mental Health', 'Pregnancy']
🧠 Classifying post for trigger warnings...
🧠 Classifying post for trigger warnings...


KeyboardInterrupt: 

In [13]:
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_prompt_2_mini.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_Prompt_2_mini.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_Prompt_2_mini.csv


In [30]:
# Cell: Batch 1-shot (k=7) few-shot labeling for 75 posts -> CSV + used examples JSON
import pandas as pd
import json, time, copy, ast, re
import numpy as np

INPUT_CSV    = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV   = "predicted_labels_75_few_shot_k_6.csv"
EXAMPLES_JSON = "used_examples_for_75_posts_K_6.json"

# --- robust parsers / helpers ---
YESNO_RE = re.compile(r"\b(YES|NO)\b", re.IGNORECASE)

def extract_yes_no(resp) -> str:
    """
    Robustly coerce model output to YES/NO.
    - Accept dicts with .get('text'), .get('content'), .get('choices'[0]['text'])
    - Take only the first line, then first YES/NO token anywhere on that line.
    """
    if resp is None:
        return "NO"
    # unwrap common response shapes
    if isinstance(resp, dict):
        if "text" in resp and isinstance(resp["text"], str):
            resp = resp["text"]
        elif "content" in resp and isinstance(resp["content"], str):
            resp = resp["content"]
        elif "choices" in resp and isinstance(resp["choices"], list) and resp["choices"]:
            ch = resp["choices"][0]
            if isinstance(ch, dict):
                if "text" in ch:
                    resp = ch["text"]
                elif "message" in ch and isinstance(ch["message"], dict) and "content" in ch["message"]:
                    resp = ch["message"]["content"]
                else:
                    resp = str(ch)
            else:
                resp = str(ch)
        else:
            resp = str(resp)
    # must be string now
    if not isinstance(resp, str):
        resp = str(resp)

    line = resp.strip().splitlines()[0] if resp.strip() else ""
    m = YESNO_RE.search(line)
    if m:
        return "YES" if m.group(1).upper() == "YES" else "NO"

    # fallback: first token check
    tok = line.upper().split()[:1]
    return "YES" if (tok and tok[0] == "YES") else "NO"

def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def tiny_throttle(sec=0.25):
    try:
        time.sleep(sec)
    except Exception:
        pass

# --- load input posts ---
df_in = pd.read_csv(INPUT_CSV)
if not {'title','body'}.issubset(df_in.columns):
    raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")

subset = df_in.head(75).copy()
subset = subset[['title','body']].fillna('').astype(str)

predicted_lists = []
all_used_examples = []   # snapshot of examples used per post

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    title = row['title']
    body  = row['body']

    # Warm up cosine neighbors (populates collected_examples_per_label)
    _ = make_multishot_prompt_cosine(
        k=6,
        title=title,
        body=body,
        current_index=None,
        include_defs=True,
        debug=False
    )

    positive_labels = []

    for j, lbl in enumerate(labels):
        single_prompt = make_prompt_for_label(
            label_name=lbl,
            k=6,
            title=title,
            body=body,
            current_index=None,
            debug=False
        )

        # First attempt (tight)
        resp = getTriggerWarningsLLMResponse(
            single_prompt,
            mode="raw",
            max_tokens=4,
            temperature=0.0,
            verbose=False
        )
        ans = extract_yes_no(resp)

        # If we didn't get a clean YES/NO on first line, retry with a slightly bigger budget
        if ans not in ("YES", "NO"):
            resp = getTriggerWarningsLLMResponse(
                single_prompt,
                mode="raw",
                max_tokens=8,   # allow newline + token quirks
                temperature=0.0,
                verbose=False
            )
            ans = extract_yes_no(resp)

        # Light logging for first 2 posts × first 3 labels to diagnose
        if i <= 2 and j < 3:
            print("\n--- DEBUG RAW ---")
            print(f"Post {i}, Label '{lbl}':")
            # Show a compact view
            if isinstance(resp, str):
                print(resp[:200].replace("\n", "\\n"))
            else:
                s = json.dumps(resp, default=str) if not isinstance(resp, str) else resp
                print(s[:200].replace("\n", "\\n"))
            print(f"Parsed: {ans}")
            print("-----------------")

        if ans == "YES":
            positive_labels.append(lbl)

        tiny_throttle(0.15)

    if not positive_labels:
        positive_labels = ["Not Applicable"]

    predicted_lists.append(positive_labels)

    snapshot = copy.deepcopy(collected_examples_per_label) if 'collected_examples_per_label' in globals() else {}
    all_used_examples.append({
        "post_index_1_based": i,
        "title": title,
        "body": body,
        "examples_by_label": snapshot
    })

    print(f"{i}. Labels: {positive_labels}")

# --- save predictions CSV ---
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\n✅ Saved predictions to {OUTPUT_CSV}")

# --- save used examples JSON ---
with open(EXAMPLES_JSON, "w", encoding="utf-8") as f:
    json.dump(all_used_examples, f, indent=2, ensure_ascii=False, default=convert_to_serializable)
print(f"✅ Saved used examples to {EXAMPLES_JSON}")



--- DEBUG RAW ---
Post 1, Label 'Medical':
Answer: YES
Parsed: YES
-----------------

--- DEBUG RAW ---
Post 1, Label 'Mental Health':
Answer: YES
Parsed: YES
-----------------

--- DEBUG RAW ---
Post 1, Label 'Abuse':
Answer: NO
Parsed: NO
-----------------
1. Labels: ['Medical', 'Mental Health', 'Pregnancy']

--- DEBUG RAW ---
Post 2, Label 'Medical':
Answer: YES
Parsed: YES
-----------------

--- DEBUG RAW ---
Post 2, Label 'Mental Health':
Answer: YES
Parsed: YES
-----------------

--- DEBUG RAW ---
Post 2, Label 'Abuse':
Answer: NO
Parsed: NO
-----------------
2. Labels: ['Medical', 'Mental Health', 'Pregnancy']
3. Labels: ['Medical', 'Pregnancy']
4. Labels: ['Medical', 'Pregnancy']
5. Labels: ['Medical', 'Mental Health', 'Pregnancy']
6. Labels: ['Medical', 'Sexual', 'Pregnancy']
7. Labels: ['Medical', 'Mental Health', 'Pregnancy']
8. Labels: ['Medical', 'Pregnancy']
9. Labels: ['Medical', 'Mental Health', 'Pregnancy']
10. Labels: ['Mental Health', 'Pregnancy']
11. Labels: ['Medi

In [31]:
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_few_shot_k_6.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_multishot_k_6.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_multishot_k_6.csv


In [17]:
# Multishot k - 1
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_1.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.747
Macro Recall: 0.905
Macro F1: 0.803

Micro Precision: 0.730
Micro Recall: 0.909
Micro F1: 0.810

Perfect Matches: 25 out of 75 posts (33.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [ ]:
# Multishot k - 2
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_2.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.741
Macro Recall: 0.914
Macro F1: 0.804

Micro Precision: 0.726
Micro Recall: 0.920
Micro F1: 0.811

Perfect Matches: 25 out of 75 posts (33.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [21]:
# Multishot k - 3
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_3.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.747
Macro Recall: 0.905
Macro F1: 0.801

Micro Precision: 0.726
Micro Recall: 0.909
Micro F1: 0.808

Perfect Matches: 23 out of 75 posts (30.7%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [22]:
# Multishot k - 4
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_4.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.751
Macro Recall: 0.913
Macro F1: 0.806

Micro Precision: 0.729
Micro Recall: 0.920
Micro F1: 0.813

Perfect Matches: 25 out of 75 posts (33.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [23]:
# Multishot k - 5
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_5.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.749
Macro Recall: 0.912
Macro F1: 0.808

Micro Precision: 0.734
Micro Recall: 0.914
Micro F1: 0.814

Perfect Matches: 25 out of 75 posts (33.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [32]:
# Multishot k - 6
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_6.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.746
Macro Recall: 0.921
Macro F1: 0.807

Micro Precision: 0.724
Micro Recall: 0.925
Micro F1: 0.812

Perfect Matches: 26 out of 75 posts (34.7%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [29]:
# Multishot k - 7
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_7.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.757
Macro Recall: 0.906
Macro F1: 0.804

Micro Precision: 0.733
Micro Recall: 0.909
Micro F1: 0.811

Perfect Matches: 25 out of 75 posts (33.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [26]:
# Multishot k - 10
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_10.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.743
Macro Recall: 0.911
Macro F1: 0.802

Micro Precision: 0.722
Micro Recall: 0.914
Micro F1: 0.807

Perfect Matches: 23 out of 75 posts (30.7%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [19]:
# 🛠️ Step 1: Clean 'Tags' column if needed
df["Tags"] = df["Tags"].apply(
    lambda x: [tag.strip() for tag in x.split(",")] if isinstance(x, str) else x
)

# 🧷 Step 2: Create binary columns per label
all_labels = [
    "Medical",
    "Mental Health",
    "Abuse",
    "Aggression",
    "Sexual",
    "Discrimination",
    "Pregnancy",
    "Not Applicable"
]

for label in all_labels:
    df[label] = df["Tags"].apply(lambda tags: int(label in tags if isinstance(tags, list) else []))


In [16]:
# # 🔹 Sample input
# sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night. I don't know if it was a miscarriage or my period. But I am very scared about it"
# title = "Relationship fight and sudden bleeding"
# body = sample_post  # or use full post if you have title separately

# # 🔹 Generate basic definition-based prompt
# prompt = make_prompt(title, body, include_defs=True)

# # 🔹 Get LLM response
# response = getTriggerWarningsLLMResponse(prompt)

# # 🔹 Show output
# print("🔍 LLM Response (basic prompt):")
# print(response)

In [20]:
print(df.columns.tolist())

['id', 'soure', 'subreddit', 'title', 'body', 'created_utc', 'url', 'Tags', 'text', 'Medical', 'Mental Health', 'Abuse', 'Aggression', 'Sexual', 'Discrimination', 'Pregnancy', 'Not Applicable']


In [21]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
title = "Bleeding After an Argument — Scared and Confused"
body = (
    "I'm really scared and don't know who else to talk to. My boyfriend and I had a horrible argument last night — he yelled at me for over an hour, and I ended up locking myself in the bathroom crying. "
    "Later that night, I noticed I was bleeding heavily, way more than my normal period, and I felt this deep cramping that wouldn't go away. "
    "I'm around 6 weeks late on my period and had been suspecting I might be pregnant, but I hadn’t taken a test yet because I was afraid of what it might say. "
    "Now, I don't know if this is a miscarriage or something else. I feel so alone. I’ve been reading online and some posts say it could just be stress-induced bleeding or maybe even implantation bleeding, but this doesn't feel like that. "
    "I don’t know whether to go to the hospital. I’m scared he might get angry again if he finds out I’m talking to anyone about this. "
    "Has anyone experienced something similar? Could stress cause this much bleeding? Or could this have been a miscarriage? I'm really not sure how to cope with everything happening right now."
)

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=title,
    body=body,
    current_index=None,
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)



['Not Applicable']


In [22]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
title = "Relationship fight and sudden bleeding"
body = "My boyfriend and I had a fight and I started bleeding heavily that night."

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=title,
    body=body,
    current_index=None,
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)


['Not Applicable']


In [73]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
# title = "Complications after abortion?"
# body = "Hi everyone, Ive read that abortions don’t cause infertility. There is a part of me that doesn’t fully believe that. Could be my anxiety speaking. Since my abortion (August 2024), I’ve been bleeding half way through my cycle and after sex, brown spotting, and recently my periods have been late (last three months) and my uterus, ovaries and cervix are so tender it hurts to walk, bend over or even apply a bit of pressure. It isn’t unbearable pain, it’s manageable but rn it’s at its worst and it’s still out of the normal for me. Anyone know what it possibly could be? I have a gynecologist appointment in about 3 weeks but I’m hurting now and I’m getting very worried."

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=df.loc[0, "title"],
    body=df.loc[0, "body"],
    current_index=0,      # <-- crucial
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)


['Medical', 'Pregnancy']


In [73]:
print(df.columns.tolist())


['id', 'soure', 'subreddit', 'title', 'body', 'created_utc', 'url', 'Tags', 'text']


In [18]:
# --- One-time init (top of notebook)
from llama_cpp import Llama
import os

MODEL_PATH = os.path.expanduser("~/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf")

# Load once and reuse (reloading every call is very slow)
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,     # fits well on t3.xlarge CPU
    n_threads=4,    # t3.xlarge has 4 vCPUs
    n_batch=128,    # adjust if you see RAM pressure
    verbose=False
)

print("✅ Local Llama loaded:", MODEL_PATH)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ Local Llama loaded: /home/ubuntu/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf


In [19]:
import time

def getTriggerWarningsLLMResponse_Llama(post_text, temperature=0, max_tokens=512, verbose=True):
    """
    Classify trigger warnings for a given post using LOCAL Llama (llama-cpp).
    Same structure as the OpenAI version, but for llama-cpp-python.

    Args:
        post_text (str): Fully-formed prompt from make_per_label_prompt().
        temperature (float): Sampling randomness.
        max_tokens (int): Max tokens in the output.
        verbose (bool): Print detailed logs.

    Returns:
        str: Raw Llama output text, or None on failure.
    """
    if verbose:
        print("🧠 Classifying post for trigger warnings (LLAMA local)...")
        print("=" * 60)

    max_retries = 3
    base_delay = 60  # seconds

    for attempt in range(max_retries):
        try:
            # Pass prompt exactly as provided
            resp = llm(
                prompt=post_text,
                temperature=float(temperature),
                max_tokens=int(max_tokens),
                top_p=1.0,
                repeat_penalty=1.1
            )
            llm_response = resp["choices"][0]["text"]

            if verbose:
                print("✅ LLM Response received:")
                print("-" * 60)

            return llm_response

        except Exception as e:
            if verbose:
                print(f"❌ Error: {e}")
            wait_time = base_delay * (2 ** attempt)
            if attempt < max_retries - 1:
                if verbose:
                    print(f"⏳ Retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
                continue
            return None

    if verbose:
        print("❌ Failed after multiple retries due to persistent issues.")
    return None

In [20]:
# Cell 3: Chain-of-thought prompting for each label (LLama backend)
import time
import pandas as pd

# Sample post for evaluation
title = "Relationship fight and sudden bleeding"
sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night."
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)

    # 🔁 Use local Llama instead of OpenAI
    response = getTriggerWarningsLLMResponse_Llama(
        single_prompt,
        temperature=0.0,
        max_tokens=16,    # small, expecting YES/NO
        verbose=False
    )

    # Normalize to strict YES/NO
    answer = (response or "NO").strip().upper()
    answer = "YES" if "YES" in answer else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []
for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")



🧠 Evaluating Label: Medical

🧠 Evaluating Label: Mental Health

🧠 Evaluating Label: Abuse

🧠 Evaluating Label: Aggression

🧠 Evaluating Label: Sexual

🧠 Evaluating Label: Discrimination

🧠 Evaluating Label: Pregnancy

🧠 Evaluating Label: Not Applicable

🔖 Final Multi-label Prediction:
Labels: ['Not Applicable']


In [22]:
# Cell 3: Chain-of-thought prompting for each label (LLama backend)
import time
import pandas as pd

# Sample post for evaluation
sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night. I don't know if it was a miscarriage or my period. But I am very scared about it"
title = "Relationship fight and sudden bleeding"
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)

    # 🔁 Use local Llama instead of OpenAI
    response = getTriggerWarningsLLMResponse_Llama(
        single_prompt,
        temperature=0.0,
        max_tokens=16,    # small, expecting YES/NO
        verbose=False
    )

    # Normalize to strict YES/NO
    answer = (response or "NO").strip().upper()
    answer = "YES" if "YES" in answer else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []
for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")




🧠 Evaluating Label: Medical

🧠 Evaluating Label: Mental Health

🧠 Evaluating Label: Abuse

🧠 Evaluating Label: Aggression

🧠 Evaluating Label: Sexual

🧠 Evaluating Label: Discrimination

🧠 Evaluating Label: Pregnancy

🧠 Evaluating Label: Not Applicable

🔖 Final Multi-label Prediction:
Labels: ['Pregnancy']


In [ ]:
# import google.generativeai as genai

# # --- Direct API Key Configuration ---

# genai.configure(api_key=API_KEY)


# def getResponseFromGemini(prompt, model="gemini-1.5-flash", temperature=0, max_tokens=1000, verbose=True):
#     """
#     Sends a prompt to the Gemini API and gets a response.
#     """
#     if verbose:
#         print("🤖 Sending prompt to Gemini API...")
#         print("=" * 60)
        
#     try:
#         generation_config = genai.types.GenerationConfig(
#             max_output_tokens=max_tokens,
#             temperature=0
#         )
#         gemini_model = genai.GenerativeModel(model)
#         response = gemini_model.generate_content(prompt, generation_config=generation_config)
        
#         llm_response = response.text
        
#         if verbose:
#             print("✅ Response received from Gemini!")
#             print("=" * 60)
#             print("LLM RESPONSE:")
#             print("-" * 40)
#             print(llm_response)
#             print("=" * 60)
            
#         return llm_response
            
#     except Exception as e:
#         if verbose:
#             print(f"❌ Error calling Gemini API: {e}")
#         return None

# # # --- Example Usage ---
# # my_prompt = "What is the purpose of the James Webb Space Telescope?"
# # gemini_response = getResponseFromGemini(my_prompt)

# # if gemini_response:
# #     print("Program continues successfully!")


In [ ]:
import google.generativeai as genai
import os

# --- Secure API Key Configuration ---
# It's recommended to load your API key from an environment variable
# for better security, rather than hardcoding it.
# API_KEY = os.environ.get("GEMINI_API_KEY")
# If you must hardcode it, replace "YOUR_API_KEY"

genai.configure(api_key=API_KEY)


def getResponseFromGemini(prompt, model="gemini-2.0-flash-lite", temperature=0, max_tokens=1000, verbose=True):
    """
    Sends a prompt to the Gemini API and gets a response.
    """
    if verbose:
        print("🤖 Sending prompt to Gemini API...")
        print("=" * 60)
        
    try:
        generation_config = {
            "temperature": temperature,
            "max_output_tokens": max_tokens,
        }
        
        # ⚙️ Define safety settings to be less restrictive
        # This is necessary for topics that might trigger the default filters.
        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
        ]

        gemini_model = genai.GenerativeModel(model)
        
        # Pass the safety settings to the generate_content method
        response = gemini_model.generate_content(
            prompt,
            generation_config=generation_config,
            safety_settings=safety_settings  
        )
        
        # --- NEW: Improved check for blocked content ---
        # This try/except block handles cases where a candidate exists but has no text,
        # which causes the `finish_reason: 8` error.
        try:
            llm_response = response.text
        except ValueError:
             if verbose:
                print("❌ Response was blocked (ValueError). Prompt feedback:")
                print(response.prompt_feedback)
             return None
        
        if verbose:
            print("✅ Response received from Gemini!")
            print("=" * 60)
            print("LLM RESPONSE:")
            print("-" * 40)
            print(llm_response)
            print("=" * 60)
            
        return llm_response
            
    except Exception as e:
        if verbose:
            print(f"❌ Error calling Gemini API: {e}")
        return None

# # --- Example Usage ---
# my_prompt = "Explain the concept of zero-shot learning in simple terms."
# gemini_response = getResponseFromGemini(my_prompt)

# if gemini_response:
#     print("Program continues successfully!")

In [ ]:
import requests
import json

# --- Together.xyz API Configuration ---
TOGETHER_API_ENDPOINT = "https://api.together.xyz/v1/chat/completions"

def getResponseFromMetaLlama(prompt, model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo", temperature=0, max_tokens=1000, verbose=True):
    """
    Sends a prompt to the Meta Llama model via Together.xyz API and gets a response.
    """
    if verbose:
        print("🤖 Sending prompt to Meta Llama API...")
        print("=" * 60)
    
    headers = {
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": max_tokens
    }
    
    try:
        response = requests.post(TOGETHER_API_ENDPOINT, headers=headers, json=data)
        
        if response.status_code == 200:
            llm_response = response.json()["choices"][0]["message"]["content"]
            
            if verbose:
                print("✅ Response received from Meta Llama!")
                print("=" * 60)
                print("LLM RESPONSE:")
                print("-" * 40)
                print(llm_response)
                print("=" * 60)
            
            return llm_response
        else:
            if verbose:
                print(f"❌ Error calling Meta Llama API: {response.status_code}")
            return None
            
    except Exception as e:
        if verbose:
            print(f"❌ Error calling Meta Llama API: {e}")
        return None

# # --- Example Usage ---
# my_prompt = "What are some fun things to do in New York?"
# meta_llama_response = getResponseFromMetaLlama(my_prompt)

# if meta_llama_response:
#     print("Program continues successfully!")


In [427]:
import time

def processRows(response_api_func, num_rows=10, start_row=0, output_file="llm_results.csv", 
                api_name="Unknown", delay=0, **api_kwargs):
    """
    Generic function to process rows using ANY LLM response API function.
    
    Args:
        response_api_func (callable): The API function to call (e.g., getResponseFromOpenAI, getResponseFromGemini)
        num_rows (int): Number of rows to process
        start_row (int): Starting row index
        output_file (str): Output CSV filename
        api_name (str): Name of the API for logging (e.g., "OpenAI GPT-4.1", "Gemini Pro")
        delay (float): Delay between API calls to avoid rate limiting
        **api_kwargs: Additional keyword arguments to pass to the API function
    
    Returns:
        pd.DataFrame or None: Results dataframe or None if failed
    """
    print(f"🚀 Processing {num_rows} rows with {api_name} starting from row {start_row}")
    print("=" * 60)
    
    results = []
    
    # Set default API parameters if not provided
    default_kwargs = {'verbose': False}
    # If using OpenAI, set default model and max_tokens for cost saving
    if response_api_func.__name__ == 'getResponseFromOpenAI':
        if 'model' not in api_kwargs:
            default_kwargs['model'] = 'gpt-4.1-2025-04-14'
        if 'max_tokens' not in api_kwargs:
            default_kwargs['max_tokens'] = 256
    default_kwargs.update(api_kwargs)
    
    for i in range(start_row, start_row + num_rows):
        if i >= len(df):
            print(f"⚠️ Reached end of dataset at row {i}")
            break
            
        print(f"📝 Processing row {i}...", end=" ")
        
        try:
            # Get data
            row = df.iloc[i]
            title = row['title']
            body = row['body']
            
            # Generate prompt
            prompt = make_prompt(title, body, include_defs=True)
            
            # Call the provided API function with the prompt and additional kwargs
            llm_response = response_api_func(prompt, **default_kwargs)
            
            if llm_response:
                # Parse to 0/1
                numeric_results = parseToNumeric(llm_response)
                
                # Create result row
                result_row = {
                    'rowNumber': i,
                    'title': title,
                    'body': body,
                    **numeric_results  # Add all label columns
                }
                results.append(result_row)
                print("✅")
            else:
                print("❌")
                
            # Delay to avoid rate limiting
            # time.sleep(delay)
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Save to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        print(f"\n💾 Results saved to {output_file}")
        print(f"📊 Processed {len(results)} rows successfully with {api_name}")
        return results_df
    else:
        print("❌ No results to save")
        return None



In [428]:
def processMultishotRows(response_api_func, k=2, num_rows=10, start_row=0, output_file="llm_multishot_results.csv", 
                         api_name="Unknown", delay=0, **api_kwargs):
    """
    Generic function to process rows using ANY LLM response API function with multi-shot prompting.
    
    Args:
        response_api_func (callable): The API function to call (e.g., getResponseFromOpenAI, getResponseFromGemini)
        k (int): Number of examples per label for few-shot prompt (default: 2)
        num_rows (int): Number of rows to process
        start_row (int): Starting row index
        output_file (str): Output CSV filename
        api_name (str): Name of the API for logging (e.g., "OpenAI GPT-4.1", "Gemini Pro")
        delay (float): Delay between API calls to avoid rate limiting
        **api_kwargs: Additional keyword arguments to pass to the API function
    
    Returns:
        pd.DataFrame or None: Results dataframe or None if failed
    """
    print(f"🚀 Processing {num_rows} rows with {api_name} (multi-shot, k={k}) starting from row {start_row}")
    print("=" * 60)
    
    results = []
    
    # Set default API parameters if not provided
    default_kwargs = {'verbose': False}
    # If using OpenAI, set default model and max_tokens for cost saving
    if response_api_func.__name__ == 'getResponseFromOpenAI':
        if 'model' not in api_kwargs:
            default_kwargs['model'] = 'gpt-4.1-2025-04-14'
        if 'max_tokens' not in api_kwargs:
            default_kwargs['max_tokens'] = 256
        # # Force minimum delay for OpenAI Tier 1 users (500 RPM = max 1 request every 0.12s)
        # if delay < 0.15:
        #     delay = 0.15  # Safe margin for Tier 1 rate limits
        #     print(f"⚠️  Setting minimum delay to {delay}s for OpenAI Tier 1 rate limits")
    
    default_kwargs.update(api_kwargs)
    
    for i in range(start_row, start_row + num_rows):
        if i >= len(df):
            print(f"⚠️ Reached end of dataset at row {i}")
            break
            
        print(f"📝 Processing row {i}...", end=" ")
        
        try:
            # Get data
            row = df.iloc[i]
            title = row['title']
            body = row['body']
            
            # Generate multi-shot prompt (key difference from processRows)
            prompt = make_multishot_prompt(k=k, title=title, body=body, current_index=i, include_defs=True)
            
            # Call the provided API function with the prompt and additional kwargs
            llm_response = response_api_func(prompt, **default_kwargs)
            
            if llm_response:
                # Parse to 0/1
                numeric_results = parseToNumeric(llm_response)
                
                # Create result row
                result_row = {
                    'rowNumber': i,
                    'title': title,
                    'body': body,
                    **numeric_results  # Add all label columns
                }
                results.append(result_row)
                print("✅")
            else:
                print("❌ Failed")
                
            # MANDATORY delay to avoid rate limiting (especially for Tier 1)
            if delay > 0:
                time.sleep(delay)
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Save to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        print(f"\n💾 Results saved to {output_file}")
        print(f"📊 Processed {len(results)} rows successfully with {api_name} (multi-shot, k={k})")
        return results_df
    else:
        print("❌ No results to save")
        return None

print("✅ processMultishotRows function updated with rate limiting!")
print("🎯 Usage: results = processMultishotRows(getResponseFromOpenAI, k=3, num_rows=50, api_name='OpenAI GPT-4.1')")
print("⚠️  Note: Automatic 0.15s delay enforced for OpenAI Tier 1 rate limits")

✅ processMultishotRows function updated with rate limiting!
🎯 Usage: results = processMultishotRows(getResponseFromOpenAI, k=3, num_rows=50, api_name='OpenAI GPT-4.1')
⚠️  Note: Automatic 0.15s delay enforced for OpenAI Tier 1 rate limits


In [429]:
# # 🚀 PROCESS ALL ROWS WITH OPENAI GPT-4.1 (0-shot)
# print("🎯 PROCESSING ALL ROWS WITH OPENAI GPT-4.1 (0-shot)")
# print("=" * 60)

# results_all_rows_openai = processRows(
#     response_api_func=getResponseFromOpenAI,
#     num_rows=len(df),
#     start_row=0,
#     output_file="all_rows_results_openai_gpt41.csv",
#     api_name="OpenAI GPT-4.1",
#     model="gpt-4.1-2025-04-14",
#     temperature=0,
#     max_tokens=256,
#     verbose=False
# )

# if results_all_rows_openai is not None:
#     print(f"\n✅ Successfully processed {len(results_all_rows_openai)} rows with OpenAI GPT-4.1!")
#     print(f"📄 Results saved to: all_rows_results_openai_gpt41.csv")
#     display_cols = ['rowNumber', 'title'] + labels
#     print(results_all_rows_openai[display_cols].head())
#     print(f"\n🎯 CALCULATING ACCURACY...")
#     accuracies_all_openai = calculateAccuracy(results_all_rows_openai, df)
#     if accuracies_all_openai and len(accuracies_all_openai) > 0:
#         avg_accuracy = sum(accuracies_all_openai.values()) / len(accuracies_all_openai) * 100
#         print(f"   • Average accuracy: {avg_accuracy:.1f}%")
#     else:
#         print(f"   • Average accuracy: Could not calculate (no matching columns)")
# else:
#     print("❌ Failed to process rows with OpenAI GPT-4.1")


In [430]:
# # 🤖 PROCESS ALL ROWS WITH GEMINI AND CALCULATE ACCURACY

# print("🎯 PROCESSING ALL ROWS WITH GEMINI")
# print("=" * 60)

# # Process all rows using the generic processRows with Gemini
# results_all_rows_gemini = processRows(
#     response_api_func=getResponseFromGemini,
#     num_rows=len(df), 
#     start_row=0, 
#     output_file="all_rows_results_gemini.csv",
#     api_name="Gemini 1.5 Flash",
#     model="gemini-1.5-flash",
#     verbose=False
# )

# # Calculate accuracy if processing was successful
# if results_all_rows_gemini is not None:
#     print(f"\n✅ Successfully processed {len(results_all_rows_gemini)} rows with Gemini!")
#     print(f"📄 Results saved to: all_rows_results_gemini.csv")
    
#     # Show sample of results
#     print(f"\n📊 SAMPLE RESULTS:")
#     display_cols = ['rowNumber', 'title'] + labels
#     print(results_all_rows_gemini[display_cols].head())
    
#     # Calculate accuracy
#     print(f"\n🎯 CALCULATING ACCURACY...")
#     accuracies_all_gemini = calculateAccuracy(results_all_rows_gemini, df)
    
#     print(f"\n📈 SUMMARY FOR ALL ROWS (GEMINI):")
#     print(f"   • Total rows processed: {len(results_all_rows_gemini)}")
#     print(f"   • Output file: all_rows_results_gemini.csv")
    
#     # Calculate average accuracy safely
#     if accuracies_all_gemini and len(accuracies_all_gemini) > 0:
#         avg_accuracy_gemini = sum(accuracies_all_gemini.values()) / len(accuracies_all_gemini) * 100
#         print(f"   • Average accuracy: {avg_accuracy_gemini:.1f}%")
#     else:
#         print(f"   • Average accuracy: Could not calculate (no matching columns)")
#         print(f"   • Available columns in results: {list(results_all_rows_gemini.columns)}")
#         print(f"   • Available columns in dataset: {list(df.columns)}")
    
# else:
#     print("❌ Failed to process rows with Gemini")

In [431]:
# # 🦙 PROCESS ALL ROWS WITH META LLAMA AND CALCULATE ACCURACY

# print("🎯 PROCESSING ALL ROWS WITH META LLAMA")
# print("=" * 60)

# # Process all rows using the generic processRows with Meta Llama
# results_all_rows_llama = processRows(
#     response_api_func=getResponseFromMetaLlama,
#     num_rows=len(df), 
#     start_row=0, 
#     output_file="all_rows_results_llama.csv",
#     api_name="Meta Llama",
#     model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
#     verbose=False
# )

# # Calculate accuracy if processing was successful
# if results_all_rows_llama is not None:
#     print(f"\n✅ Successfully processed {len(results_all_rows_llama)} rows with Meta Llama!")
#     print(f"📄 Results saved to: all_rows_results_llama.csv")
    
#     # Show sample of results
#     print(f"\n📊 SAMPLE RESULTS:")
#     display_cols = ['rowNumber', 'title'] + labels
#     print(results_all_rows_llama[display_cols].head())
    
#     # Calculate accuracy
#     print(f"\n🎯 CALCULATING ACCURACY...")
#     accuracies_all_llama = calculateAccuracy(results_all_rows_llama, df)
    
#     print(f"\n📈 SUMMARY FOR ALL ROWS (META LLAMA):")
#     print(f"   • Total rows processed: {len(results_all_rows_llama)}")
#     print(f"   • Output file: all_rows_results_llama.csv")
    
#     # Calculate average accuracy safely
#     if accuracies_all_llama and len(accuracies_all_llama) > 0:
#         avg_accuracy_llama = sum(accuracies_all_llama.values()) / len(accuracies_all_llama) * 100
#         print(f"   • Average accuracy: {avg_accuracy_llama:.1f}%")
#     else:
#         print(f"   • Average accuracy: Could not calculate (no matching columns)")
#         print(f"   • Available columns in results: {list(results_all_rows_llama.columns)}")
#         print(f"   • Available columns in dataset: {list(df.columns)}")
    
# else:
#     print("❌ Failed to process rows with Meta Llama")

In [432]:
# # 🚀 PROCESS ALL 985 ROWS WITH OPENAI GPT-4.1 (k=2, Long Context Tier 1)
# print("🎯 MULTISHOT LEARNING: OpenAI GPT-4.1, k=2, Long Context Tier 1, ALL ROWS")
# print("=" * 60)
# results_openai_multishot_k2_all = processMultishotRows(
#     getResponseFromOpenAI,
#     k=2,
#     num_rows=985,
#     start_row=0,
#     output_file="openai_multishot_k2_longcontext_allrows.csv",
#     api_name="OpenAI GPT-4.1 (k=2, Long Context Tier 1, ALL ROWS)",
#     delay=0.8,
#     max_tokens=1500,
#     model="gpt-4.1-2025-04-14",
#     verbose=False
# )

In [433]:
# # 📈 CALCULATE ACCURACY FOR OPENAI MULTISHOT (k=2, ALL ROWS)
# if results_openai_multishot_k2_all is not None:
#     print("\n✅ Successfully processed all 985 rows with OpenAI GPT-4.1 (k=2)!")
#     print(f"📄 Results saved to: openai_multishot_k2_longcontext_allrows.csv")
#     display_cols = ['rowNumber', 'title'] + labels
#     print(results_openai_multishot_k2_all[display_cols].head())
#     print(f"\n🎯 CALCULATING ACCURACY...")
#     accuracies_k2_all = calculateAccuracy(results_openai_multishot_k2_all, df)
#     if accuracies_k2_all and len(accuracies_k2_all) > 0:
#         avg_accuracy_k2_all = sum(accuracies_k2_all.values()) / len(accuracies_k2_all) * 100
#         print(f"   • Average accuracy: {avg_accuracy_k2_all:.1f}%")
#     else:
#         print(f"   • Average accuracy: Could not calculate (no matching columns)")
# else:
#     print("❌ Failed to process all rows with OpenAI GPT-4.1 (k=2)")

In [434]:
# 🔍 Print full prompt and LLM output for k=2, row 0 and row 1
text_samples = []
for row_idx in range(0,100):
    print(f"\n==================== ROW {row_idx} ====================")
    # Get the title and body for this row
    row = df.iloc[row_idx]
    title = row['title']
    body = row['body']
    # Generate the full prompt as used in multishot
    prompt = make_multishot_prompt(k=8, title=title, body=body, current_index=row_idx, include_defs=True)
    # print("\n--- FULL PROMPT SENT TO LLM ---\n")
    # print(prompt)
    text_samples.append(prompt)
    # print("\n--- END OF PROMPT ---\n")
    # Get the LLM output from results if available
    # if results_openai_multishot_k2_all is not None and row_idx in results_openai_multishot_k2_all['rowNumber'].values:
    #     result_row = results_openai_multishot_k2_all[results_openai_multishot_k2_all['rowNumber'] == row_idx]
    #     if 'llm_response' in result_row.columns:
    #         print("\n--- LLM RAW OUTPUT ---\n")
    #         print(result_row['llm_response'].values[0])
    #         print("\n--- END OF LLM OUTPUT ---\n")
    #     else:
    #         print("\n(No raw LLM response stored; showing parsed labels)")
    #         for label in labels:
    #             print(f"{label}: {result_row[label].values[0]}")
    # else:
    #     print("\n(No LLM output found for this row in results)")


==================== ROW 0 ====================

==================== ROW 1 ====================

==================== ROW 2 ====================

==================== ROW 3 ====================

==================== ROW 4 ====================

==================== ROW 5 ====================

==================== ROW 6 ====================

==================== ROW 7 ====================

==================== ROW 8 ====================

==================== ROW 9 ====================

==================== ROW 10 ====================

==================== ROW 11 ====================

==================== ROW 12 ====================

==================== ROW 13 ====================

==================== ROW 14 ====================

==================== ROW 15 ====================

==================== ROW 16 ====================

==================== ROW 17 ====================

==================== ROW 18 ====================

==================== ROW 19 ====================

=========

In [435]:
# in your notebook
%pip install tiktoken

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # or another model encoding
sample = "This is a test string to count tokens."
print(len(enc.encode(sample)))



[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
9


In [436]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")  # For GPT-4/3.5; check your model
token_counts = [len(enc.encode(sample)) for sample in text_samples]  # text_samples is your list of 100 strings

# Compute stats
average_token_size = sum(token_counts) / len(token_counts)
median_token_size = sorted(token_counts)[len(token_counts) // 2]
print("Average:", average_token_size, "Median:", median_token_size)

Average: 15156.71 Median: 15042


In [437]:
# # 1. Install 🤗 Transformers if you haven’t already
# #    In your notebook:
# #    %pip install transformers
# %pip install transformers
# from transformers import AutoTokenizer

# # 2. Load the Qwen-3-1.7B tokenizer
# #    (the “trust_remote_code=True” flag lets HF pull in any custom code needed)
# tokenizer = AutoTokenizer.from_pretrained(
#     "Qwen/Qwen-3-1.7B",
#     trust_remote_code=True
# )

# # 3. Tokenize your samples and compute stats
# token_counts = [len(tokenizer(sample)["input_ids"]) for sample in text_samples]

# average_token_size = sum(token_counts) / len(token_counts)
# median_token_size = sorted(token_counts)[len(token_counts) // 2]

# print("Average:", average_token_size)
# print("Median:", median_token_size)


In [438]:
text_samples[4]

'You are conducting objective research analysis of sexual violence stories. Please evaluate each consent criterion based on the provided text and examples.\n\nIMPORTANT: Only mark a criterion as violated (YES) if the language in the story suggests that the specific standard was not met. If information is ambiguous, unclear, or missing, answer NO.\n\nEvaluate these consent criteria:\n1. Freely given\n2. Reversible\n3. Informed\n4. Enthusiastic\n5. Explicit\n6. Specific\n7. Unknown\n8. Non-physical Harrasment\nConsent Standards (mark as violated only if explicitly contradicted):\n• Freely given: All participants should have a choice to make without pressure, manipulation, or under the influence of drugs or alcohol. Moreover, minors are not able to consent.\n• Reversible: During sexual activity, all participants can change their minds anytime and be free to stop the sexual activity.\n• Informed: All participants should be aware of all facts and information about the sexual activity before

In [439]:
# Debug: Show which rows are selected/dropped for k=5 in multishot prompt (first 10 rows)

def make_multishot_prompt_debug(k, title, body, current_index=None, include_defs=True):
    """
    Same as make_multishot_prompt, but prints which rows are selected/dropped for each label.
    """
    instr = (
        "You are conducting objective research analysis of sexual violence stories. "
        "Please evaluate each consent criterion based on the provided text and examples.\n\n"
        "IMPORTANT: Only mark a criterion as violated (YES) if the language in the story suggests that the specific standard was not met. "
        "If information is ambiguous, unclear, or missing, answer NO.\n\n"
        "Evaluate these consent criteria:\n"
    )
    for i, label in enumerate(labels, 1):
        instr += f"{i}. {label}\n"
    defs_text = ""
    if include_defs:
        defs_text = "Consent Standards (mark as violated only if explicitly contradicted):\n" + "\n".join(
            f"• {l}: {v}" for l, v in definitions.items()
        ) + "\n\n"
    examples_text = "--- EXAMPLES ---\n"
    debug_info = {}
    for label in labels:
        if label == "Non-physical Harrasment":
            actual_col = "Non-physical Harrasment"
        else:
            actual_col = f"{label} violation"
        violation_examples = df[df[actual_col] == 1]
        all_indices = list(violation_examples.index)
        dropped = []
        if current_index is not None and current_index in violation_examples.index:
            violation_examples = violation_examples.drop(current_index)
            dropped = [current_index]
        after_drop_indices = list(violation_examples.index)
        if len(violation_examples) > 0:
            num_samples = min(k, len(violation_examples))
            sampled_rows = violation_examples.sample(n=num_samples, random_state=42)
            sampled_indices = list(sampled_rows.index)
        else:
            sampled_indices = []
        debug_info[label] = {
            'all_indices': all_indices,
            'dropped': dropped,
            'after_drop_indices': after_drop_indices,
            'sampled_indices': sampled_indices
        }
        # (rest of prompt construction omitted for debug)
    return debug_info

# Run for first 10 rows and print debug info
for row_idx in range(10):
    print(f"\n==================== ROW {row_idx} ====================")
    row = df.iloc[row_idx]
    title = row['title']
    body = row['body']
    debug = make_multishot_prompt_debug(k=5, title=title, body=body, current_index=row_idx, include_defs=True)
    for label in labels:
        info = debug[label]
        print(f"Label: {label}")
        print(f"  All possible example indices: {info['all_indices']}")
        if info['dropped']:
            print(f"  Dropped (current row): {info['dropped']}")
        print(f"  After drop: {info['after_drop_indices']}")
        print(f"  Sampled for prompt: {info['sampled_indices']}")
    print("-"*50)


==================== ROW 0 ====================
Label: Freely given
  All possible example indices: [0, 5, 6, 8, 9, 10, 15, 21, 22, 23, 26, 27, 30, 41, 50, 51, 61, 66, 67, 71, 72, 73, 74, 81, 82, 84, 86, 87, 89, 90, 92, 93, 94, 96, 97, 98, 99, 100, 101, 103, 107, 110, 111, 113, 116, 117, 118, 119, 120, 122, 124, 125, 127, 128, 129, 133, 134, 136, 137, 139, 141, 153, 155, 156, 157, 158, 163, 164, 165, 167, 169, 173, 174, 175, 176, 178, 181, 183, 186, 187, 188, 190, 193, 195, 196, 197, 198, 200, 203, 206, 207, 210, 211, 212, 213, 214, 215, 216, 219, 221, 223, 224, 227, 228, 229, 230, 231, 232, 233, 234, 236, 238, 239, 240, 246, 247, 248, 249, 250, 251, 252, 254, 265, 268, 269, 271, 272, 274, 275, 276, 278, 279, 281, 282, 283, 284, 287, 288, 289, 290, 291, 292, 295, 296, 300, 301, 307, 308, 309, 310, 311, 313, 314, 316, 318, 319, 320, 321, 326, 327, 328, 329, 331, 335, 336, 337, 338, 339, 340, 346, 347, 349, 350, 354, 355, 356, 358, 359, 360, 362, 363, 364, 366, 371, 375, 376, 377, 379, 

In [440]:
# 🚀 PROCESS ALL ROWS WITH GEMINI (k=8, gemini-2.0-flash-lite)
print("🎯 PROCESSING ALL ROWS WITH GEMINI (k=8, gemini-2.0-flash-lite)")
print("=" * 60)

results_all_rows_gemini_k8 = processMultishotRows(
    response_api_func=getResponseFromGemini,
    k=3,
    num_rows=len(df),
    start_row=0,
    output_file="gemini_multishot_k8_flashlite_results.csv",
    api_name="Gemini 2.0 Flash Lite (k=8)",
    model="gemini-2.0-flash-lite",
    verbose=True,
    delay=1
)

if results_all_rows_gemini_k8 is not None:
    print(f"\n✅ Successfully processed {len(results_all_rows_gemini_k8)} rows with Gemini 2.0 Flash Lite (k=8)!")
    print(f"📄 Results saved to: gemini_multishot_k8_flashlite_results.csv")
    display_cols = ['rowNumber', 'title'] + labels
    print(results_all_rows_gemini_k8[display_cols].head())
    print(f"\n🎯 CALCULATING ACCURACY...")
    accuracies_gemini_k8 = calculateAccuracy(results_all_rows_gemini_k8, df)
    if accuracies_gemini_k8 and len(accuracies_gemini_k8) > 0:
        avg_accuracy_gemini_k8 = sum(accuracies_gemini_k8.values()) / len(accuracies_gemini_k8) * 100
        print(f"   • Average accuracy: {avg_accuracy_gemini_k8:.1f}%")
    else:
        print(f"   • Average accuracy: Could not calculate (no matching columns)")
        print(f"   • Available columns in results: {list(results_all_rows_gemini_k8.columns)}")
        print(f"   • Available columns in dataset: {list(df.columns)}")
else:
    print("❌ Failed to process rows with Gemini 2.0 Flash Lite (k=8)")

🎯 PROCESSING ALL ROWS WITH GEMINI (k=8, gemini-2.0-flash-lite)
🚀 Processing 989 rows with Gemini 2.0 Flash Lite (k=8) (multi-shot, k=3) starting from row 0
📝 Processing row 0... 🤖 Sending prompt to Gemini API...
✅ Response received from Gemini!
LLM RESPONSE:
----------------------------------------
Here's the analysis of the story based on the provided consent criteria:

Freely given: NO
Reversible: NO
Informed: NO
Enthusiastic: NO
Explicit: NO
Specific: NO
Unknown: NO
Non-physical Harrasment: NO

✅
📝 Processing row 1... 🤖 Sending prompt to Gemini API...
❌ Response was blocked (ValueError). Prompt feedback:
block_reason: PROHIBITED_CONTENT

❌ Failed
📝 Processing row 2... 🤖 Sending prompt to Gemini API...
❌ Response was blocked (ValueError). Prompt feedback:
block_reason: PROHIBITED_CONTENT

❌ Failed
📝 Processing row 3... 🤖 Sending prompt to Gemini API...
❌ Response was blocked (ValueError). Prompt feedback:
block_reason: PROHIBITED_CONTENT

❌ Failed
📝 Processing row 4... 🤖 Sending prom

KeyboardInterrupt: 